In [ ]:
import os
from collections import Counter
from datetime import datetime, timedelta
import pandas as pd
from typing import Dict, Tuple


In [ ]:
DATASETS = ["Books", "Electronics", "Sports", "Tools"]
DATASET_NAME = "".join([d[0] for d in DATASETS])
OUTPUT_DIR = f"path/to/SyNCRec/dataset/{DATASET_NAME}"

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
def _load_item_maps(dataset_name: str) -> Tuple[Dict[str, str], Dict[str, str]]:
    item_path = f"./{dataset_name}/{dataset_name}.item"
    if not os.path.exists(item_path):
        print(f"Error: Item file not found at {item_path}")
        return {}, {}

    with open(item_path, "r") as f:
        item_lines = [line.strip() for line in f.readlines()]
        item_id_map = {str(i + 1): item_lines[i] for i in range(len(item_lines))}
        item_domain_map = {
            str(i + 1): line.split("_")[-1] for (i, line) in enumerate(item_lines)
        }
    return item_id_map, item_domain_map

# 2. Load item ID mappings
item_id_map, item_domain_map = _load_item_maps(DATASET_NAME)


In [ ]:
def _process_sequences(
    dataset_name: str, item_id_map: Dict[str, str], item_domain_map: Dict[str, str]
) -> Tuple[pd.DataFrame, Dict[str, Counter]]:
    train_path = f"./{dataset_name}/{dataset_name}.full.test.inter"
    if not os.path.exists(train_path):
        print(f"Error: Train file not found at {train_path}")
        return pd.DataFrame(), {}

    df = pd.read_csv(
        train_path,
        sep="\t",
        header=0,
        names=["user_id", "item_seq", "target_item"],
        dtype={"user_id": str, "item_seq": str, "target_item": str},
    )

    # Create a list of all item IDs
    df["all_items"] = df.apply(
        lambda row: row["item_seq"].split() + [row["target_item"]], axis=1
    )

    # Explode the list and map the attributes
    df_expanded = df.explode("all_items").reset_index()
    df_expanded.rename(
        columns={"all_items": "item_id", "index": "original_index"}, inplace=True
    )

    df_expanded["raw_item"] = df_expanded["item_id"].map(item_id_map)
    df_expanded["type"] = df_expanded["item_id"].map(item_domain_map)

    # Create timestamps based on sequence order
    base_time = datetime(2025, 1, 1, 0, 0, 0)
    time_interval = timedelta(hours=1)
    df_expanded["unix_time"] = (
        df_expanded.groupby("original_index")
        .cumcount()
        .apply(lambda i: int((base_time + i * time_interval).timestamp()))
    )

    # Aggregate back into sequences
    amazon_seq_df = (
        df_expanded.groupby("user_id", sort=False)
        .agg(
            item=("raw_item", lambda x: ",".join(x)),
            type=("type", lambda x: ",".join(x)),
            unix_time=("unix_time", lambda x: ",".join(map(str, x))),
            list_len=("raw_item", "count"),
        )
        .reset_index()
    )

    # Count occurrences for vocabularies
    item_counter = Counter(df_expanded["raw_item"])
    # type_counter = Counter(df_expanded["type"])

    unique_items_df = df_expanded[["raw_item", "type"]].drop_duplicates()
    type_counter = Counter(unique_items_df["type"])

    return amazon_seq_df, {
        "item": item_counter,
        "type": type_counter,
    }


# 3. Process user sequences and generate vocabularies
amazon_seq_df, vocabs = _process_sequences(DATASET_NAME, item_id_map, item_domain_map)


In [ ]:
def _save_dataframes(
    amazon_seq_df: pd.DataFrame,
    vocabs: Dict[str, Counter],
    output_dir: str,
    dataset_name: str,
):
    # Save sequence data
    amazon_seq_path = os.path.join(output_dir, f"{dataset_name}_seq.txt")
    amazon_seq_df.to_csv(amazon_seq_path, sep="\t", index=True)
    print(f"Saved sequences  to {amazon_seq_path}")

    # Helper function to save vocabulary DataFrames
    def save_vocab(name: str, counter: Counter, sort_key=None):
        df = pd.DataFrame({name: list(counter.keys()), "cnt": list(counter.values())})
        if sort_key:
            df = df.sort_values(name, key=sort_key, ignore_index=True)
        else:
            df = df.sort_values(name, ignore_index=True)
        path = os.path.join(output_dir, f"voca_{name}.txt")
        df.to_csv(path, sep="\t", index=True)
        print(f"Saved vocabulary to {path}")

    save_vocab("item", vocabs["item"], sort_key=lambda x: x.str.split("_").str[-1])
    save_vocab("type", vocabs["type"])


# 4. Save the results
_save_dataframes(amazon_seq_df, vocabs, OUTPUT_DIR, DATASET_NAME)
print("\nData processing complete.")
